# 02 — Preprocessing & Data Augmentation

Validates the augmentation pipeline and class-balancing strategy.  
CLAHE preprocessing is specifically important for chest X-rays.

### Objectives
1. Validate CLAHE preprocessing effect on X-ray quality
2. Visualize augmentation pipeline output
3. Verify WeightedRandomSampler balances class distribution
4. Confirm DataLoader batch statistics

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
from pathlib import Path

from src.data import ChestXRayDataset, build_dataloaders
from src.data.dataset import get_transforms, IDX_TO_CLASS

plt.rcParams['figure.dpi'] = 120
DATA_ROOT = '../chest_xray'
random.seed(42)
np.random.seed(42)

## 1. CLAHE Effect on X-Ray Images

In [ ]:
from pathlib import Path
import albumentations as A

# Sample one normal and one pneumonia image
normal_img_path    = next((Path(DATA_ROOT) / 'train' / 'NORMAL').glob('*.jpeg'))
pneumonia_img_path = next((Path(DATA_ROOT) / 'train' / 'PNEUMONIA').glob('*bacteria*'))

def load_rgb(path):
    img = cv2.imread(str(path))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def apply_clahe_rgb(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
samples = [(normal_img_path, 'NORMAL'), (pneumonia_img_path, 'PNEUMONIA (Bacterial)')]

for row, (path, label) in enumerate(samples):
    orig = load_rgb(path)
    enhanced = apply_clahe_rgb(orig)
    orig_resized = cv2.resize(orig, (224, 224))
    enhanced_resized = cv2.resize(enhanced, (224, 224))
    diff = cv2.absdiff(orig_resized, enhanced_resized)
    
    axes[row,0].imshow(orig_resized, cmap='gray'); axes[row,0].set_title(f'Original — {label}', fontweight='bold'); axes[row,0].axis('off')
    axes[row,1].imshow(enhanced_resized, cmap='gray'); axes[row,1].set_title('After CLAHE', fontweight='bold'); axes[row,1].axis('off')
    axes[row,2].imshow(diff[:,:,0], cmap='hot'); axes[row,2].set_title('Difference Map', fontweight='bold'); axes[row,2].axis('off')

plt.suptitle('CLAHE Preprocessing — Contrast Enhancement for Chest X-Rays', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/preprocessing_clahe.png', bbox_inches='tight', dpi=150)
plt.show()

## 2. Augmentation Pipeline Visualization

In [ ]:
import albumentations as A

# Load a single image and apply augmentation 8 times
img_path = pneumonia_img_path
orig_img = cv2.imread(str(img_path))
orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)

train_transform = get_transforms('train', 224)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes[0, 0].imshow(cv2.resize(orig_img, (224, 224)), cmap='gray')
axes[0, 0].set_title('Original', fontweight='bold', color='navy')
axes[0, 0].axis('off')

for i, ax in enumerate(axes.flatten()[1:]):
    augmented = train_transform(image=orig_img)['image']
    # Denormalize for display
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_display = augmented.permute(1, 2, 0).numpy() * std + mean
    img_display = np.clip(img_display, 0, 1)
    ax.imshow(img_display, cmap='gray')
    ax.set_title(f'Augmented #{i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle('Data Augmentation Pipeline — PNEUMONIA Sample\n(HFlip, Rotation, Brightness, Grid Distortion, CoarseDropout, CLAHE)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/preprocessing_augmentation.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. WeightedRandomSampler Validation

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    DATA_ROOT, image_size=224, batch_size=32, num_workers=0, use_weighted_sampler=True
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

# Check class distribution in first 10 batches
label_counts = {0: 0, 1: 0}
for i, (imgs, labels) in enumerate(train_loader):
    for lbl in labels.numpy():
        label_counts[lbl] += 1
    if i >= 9:
        break

total = sum(label_counts.values())
print(f'\nFirst 10 batches class distribution:')
print(f'  NORMAL:    {label_counts[0]} ({label_counts[0]/total:.1%})')
print(f'  PNEUMONIA: {label_counts[1]} ({label_counts[1]/total:.1%})')
print(f'  → Sampler working correctly if ~50/50 distribution')

In [ ]:
# Verify batch tensor shape and statistics
imgs, labels = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}  (B x C x H x W)')
print(f'Dtype: {imgs.dtype}')
print(f'Value range: [{imgs.min():.3f}, {imgs.max():.3f}]')
print(f'Labels: {labels[:16].tolist()}')
print(f'Label distribution: NORMAL={( labels==0).sum()}, PNEUMONIA={(labels==1).sum()}')